# 02 — Data Cleaning & ETL Pipeline

**Steps:** Load raw data → Drop non-analytical columns → Handle missing values → Standardize brands → Fix dtypes → Flag anomalies → Remove duplicates → Feature engineering → Validate → Export

---
## 1. Imports & Configuration

In [27]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
print('Libraries loaded.')

Libraries loaded.


---
## 2. Load Raw Dataset

In [30]:
raw_df = pd.read_csv('../data/raw/BigBasket_Products.csv')
df = raw_df.copy()
print(f'Raw dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
df.head()

Raw dataset shape: (27555, 10)
Rows: 27,555  |  Columns: 10


,index,product,category,sub_category,brand,sale_price,market_price,type,rating,description
0,1,Garlic Oil - Vegetarian Capsule 500 mg,Beauty & Hygiene,Hair Care,Sri Sri Ayurveda,220.00,220.00,Hair Oil & Serum,4.10,This Product contains Garlic Oil that is known...
1,2,Water Bottle - Orange,"Kitchen, Garden & Pets",Storage & Accessories,Mastercook,180.00,180.00,Water & Fridge Bottles,2.30,"Each product is microwave safe (without lid), ..."
2,3,"Brass Angle Deep - Plain, No.2",Cleaning & Household,Pooja Needs,Trm,119.00,250.00,Lamp & Lamp Oil,3.40,"A perfect gift for all occasions, be it your m..."
3,4,Cereal Flip Lid Container/Storage Jar - Assort...,Cleaning & Household,Bins & Bathroom Ware,Nakoda,149.00,176.00,"Laundry, Storage Baskets",3.70,Multipurpose container with an attractive desi...
4,5,Creme Soft Soap - For Hands & Body,Beauty & Hygiene,Bath & Hand Wash,Nivea,162.00,162.00,Bathing Bars & Soaps,4.40,Nivea Creme Soft Soap gives your skin the best...


In [32]:
print('Missing Values')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
pd.DataFrame({'Count': missing, '%': missing_pct}).query('Count > 0')

Missing Values


,Count,%
product,1,0.00
brand,1,0.00
rating,8626,31.30
description,115,0.42


---
## 3. Drop Non-Analytical Columns

The `index` column is just a row identifier.

In [33]:
df = df.drop(columns=['index'])
print(f'Dropped: index column | Remaining columns: {df.shape[1]}')
print(df.columns.tolist())

Dropped: index column | Remaining columns: 9
['product', 'category', 'sub_category', 'brand', 'sale_price', 'market_price', 'type', 'rating', 'description']


---
## 4. Handle Missing Values

| Column | Nulls | Strategy |
|--------|-------|----------|
| `rating` | ~8,626 (31.3%) | Impute with category-wise median |
| `description` | 115 | Placeholder text |
| `product` | 1 | "Unknown Product" |
| `brand` | 1 | "Unknown" |

In [39]:
print('BEFORE:')
print(df.isnull().sum())

# Handling missing categorical/text data
df['product'] = df['product'].fillna('Unknown Product')
df['brand'] = df['brand'].fillna('Unknown')
df['description'] = df['description'].fillna('No description available')

# Handling missing numerical data
df['rating'] = df.groupby('category')['rating'].transform(lambda x: x.fillna(x.median()))
# fill with global median of ratings
global_median = raw_df['rating'].median()
df['rating'] = df['rating'].fillna(global_median)

print(f'\nAFTER:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

BEFORE:
product             0
category            0
sub_category        0
brand               0
sale_price          0
market_price        0
type                0
rating              0
description         0
is_price_anomaly    0
discount_amount     0
discount_pct        0
dtype: int64

AFTER:
product             0
category            0
sub_category        0
brand               0
sale_price          0
market_price        0
type                0
rating              0
description         0
is_price_anomaly    0
discount_amount     0
discount_pct        0
dtype: int64

Total missing: 0


---
## 5. Standardize Brand Names

Strip whitespace + convert to Title Case.

In [40]:
print(f'Unique brands BEFORE: {df["brand"].nunique()}')
df['brand'] = df['brand'].str.strip().str.title()
print(f'Unique brands AFTER:  {df["brand"].nunique()}')

Unique brands BEFORE: 2313
Unique brands AFTER:  2313


---
## 6. Fix Data Types

Ensure price columns are float64. Handle any `₹` symbols or commas if present.

In [41]:
for col in ['sale_price', 'market_price']:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.replace('₹', '', regex=False).str.replace(',', '', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f'Converted {col} to numeric')
    else:
        print(f'{col} already numeric ({df[col].dtype})')
df['sale_price'] = df['sale_price'].astype('float64')
df['market_price'] = df['market_price'].astype('float64')

sale_price already numeric (float64)
market_price already numeric (float64)


---
## 7. Flag Data Anomalies & Remove Duplicates

In [42]:
# Flag price anomalies
df['is_price_anomaly'] = df['sale_price'] > df['market_price']
print(f'Price anomalies (sale > market): {df["is_price_anomaly"].sum()}')

# Remove duplicates
dup_count = df.duplicated().sum()
print(f'Duplicates found: {dup_count}')
if dup_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'After removal: {df.shape}')
print(f'Dataset shape: {df.shape}')

Price anomalies (sale > market): 0
Duplicates found: 0
Dataset shape: (27200, 12)


---
## 8. Feature Engineering

| Feature | Formula |
|---|---|
| `discount_amount` | `market_price − sale_price` |
| `discount_pct` | `(discount_amount / market_price) × 100` |
| `price_segment` | Budget (≤₹100) / Mid-range (₹100–₹500) / Premium (>₹500) |
| `is_discounted` | True if `sale_price < market_price` |
| `rating_segment` | Low (<3) / Medium (3–4) / High (>4) |

In [43]:
# Discount Amount & Percentage
df['discount_amount'] = (df['market_price'] - df['sale_price']).round(2)
df['discount_pct'] = np.where(
    df['market_price'] > 0,
    ((df['market_price'] - df['sale_price']) / df['market_price'] * 100).round(2),
    0.0
)
print('discount_pct stats:')
print(df['discount_pct'].describe())

discount_pct stats:
count   27200.00
mean       11.85
std        14.65
min         0.00
25%         0.00
50%         5.00
75%        20.00
max        83.67
Name: discount_pct, dtype: float64


In [44]:
# Price Segment
df['price_segment'] = pd.cut(
    df['sale_price'],
    bins=[-np.inf, 100, 500, np.inf],
    labels=['Budget', 'Mid-range', 'Premium']
)
print('Price segment distribution:')
print(df['price_segment'].value_counts())
print()
print((df['price_segment'].value_counts(normalize=True) * 100).round(2))

Price segment distribution:
price_segment
Mid-range    15449
Budget        7717
Premium       4034
Name: count, dtype: int64

price_segment
Mid-range   56.80
Budget      28.37
Premium     14.83
Name: proportion, dtype: float64


In [45]:
# Is Discounted
df['is_discounted'] = df['sale_price'] < df['market_price']
print(f'Discounted: {df["is_discounted"].sum()} ({df["is_discounted"].mean()*100:.1f}%)')

Discounted: 15046 (55.3%)


In [46]:
# Rating Segment
df['rating_segment'] = pd.cut(
    df['rating'],
    bins=[-np.inf, 3, 4, np.inf],
    labels=['Low', 'Medium', 'High']
)
print('Rating segment distribution:')
print(df['rating_segment'].value_counts())
print()
print((df['rating_segment'].value_counts(normalize=True) * 100).round(2))

Rating segment distribution:
rating_segment
High      17452
Medium     7719
Low        2029
Name: count, dtype: int64

rating_segment
High     64.16
Medium   28.38
Low       7.46
Name: proportion, dtype: float64


---
## 9. Post-Cleaning Validation

In [ ]:
print('POST CLEANING VALIDATION REPORT')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'\nColumns ({df.shape[1]}):')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col} ({df[col].dtype})')
print()
df.describe()

POST-CLEANING VALIDATION REPORT
Shape: 27,200 rows × 15 columns
Missing values: 0
Duplicates: 0

Columns (15):
   1. product (str)
   2. category (str)
   3. sub_category (str)
   4. brand (str)
   5. sale_price (float64)
   6. market_price (float64)
   7. type (str)
   8. rating (float64)
   9. description (str)
  10. is_price_anomaly (bool)
  11. discount_amount (float64)
  12. discount_pct (float64)
  13. price_segment (category)
  14. is_discounted (bool)
  15. rating_segment (category)



,sale_price,market_price,rating,discount_amount,discount_pct
count,27200.00,27200.00,27200.00,27200.00,27200.00
mean,320.95,380.36,3.99,59.42,11.85
std,486.90,582.37,0.61,170.01,14.65
min,2.45,3.00,1.00,0.00,0.00
25%,95.00,100.00,3.90,0.00,0.00
50%,190.00,220.00,4.10,6.00,5.00
75%,351.00,425.00,4.20,51.00,20.00
max,12500.00,12500.00,5.00,4320.00,83.67


In [49]:
# Preview engineered features
cols = ['product', 'sale_price', 'market_price', 'discount_amount',
        'discount_pct', 'price_segment', 'is_discounted', 'rating', 'rating_segment']
df[cols].head(10)

,product,sale_price,market_price,discount_amount,discount_pct,price_segment,is_discounted,rating,rating_segment
0,Garlic Oil - Vegetarian Capsule 500 mg,220.00,220.00,0.00,0.00,Mid-range,False,4.10,High
1,Water Bottle - Orange,180.00,180.00,0.00,0.00,Mid-range,False,2.30,Low
2,"Brass Angle Deep - Plain, No.2",119.00,250.00,131.00,52.40,Mid-range,True,3.40,Medium
3,Cereal Flip Lid Container/Storage Jar - Assort...,149.00,176.00,27.00,15.34,Mid-range,True,3.70,Medium
4,Creme Soft Soap - For Hands & Body,162.00,162.00,0.00,0.00,Mid-range,False,4.40,High
5,Germ - Removal Multipurpose Wipes,169.00,199.00,30.00,15.08,Mid-range,True,3.30,Medium
6,Multani Mati,58.00,58.00,0.00,0.00,Budget,False,3.60,Medium
7,Hand Sanitizer - 70% Alcohol Base,250.00,250.00,0.00,0.00,Mid-range,False,4.00,Medium
8,Biotin & Collagen Volumizing Hair Shampoo + Bi...,1098.00,1098.00,0.00,0.00,Premium,False,3.50,Medium
9,"Scrub Pad - Anti- Bacterial, Regular",20.00,20.00,0.00,0.00,Budget,False,4.30,High


---
## 10. Export Cleaned Dataset

In [51]:
output_path = '../data/processed/bigbasket_cleaned.csv'
df.to_csv(output_path, index=False)

verify = pd.read_csv(output_path)
print(f'Exported to: {output_path}')
print(f'   Shape: {verify.shape}')
print(f'   Columns: {verify.columns.tolist()}')

Exported to: ../data/processed/bigbasket_cleaned.csv
   Shape: (27200, 15)
   Columns: ['product', 'category', 'sub_category', 'brand', 'sale_price', 'market_price', 'type', 'rating', 'description', 'is_price_anomaly', 'discount_amount', 'discount_pct', 'price_segment', 'is_discounted', 'rating_segment']


---
## Cleaning Summary

| Step | Action | Details |
|------|--------|---------|
| 1 | Dropped `index` column | Non-analytical row identifier |
| 2 | Filled `product` nulls | 1 row → "Unknown Product" |
| 3 | Filled `brand` nulls | 1 row → "Unknown" |
| 4 | Filled `description` nulls | 115 rows → "No description available" |
| 5 | Imputed `rating` nulls | 8,626 rows → category-wise median |
| 6 | Standardized brands | Stripped whitespace + title case |
| 7 | Validated price dtypes | Ensured float64 |
| 8 | Flagged anomalies | `is_price_anomaly` column |
| 9 | Removed duplicates | 355 duplicate rows removed |
| 10 | Engineered 6 features | discount_amount, discount_pct, price_segment, is_discounted, rating_segment, is_price_anomaly |

**Output:** `data/processed/bigbasket_cleaned.csv`